# Bolyai Labs – Feladatkártya #3: Sikeres lesz-e a hívás?

> Termékfelelősként szeretném tudni előre, melyik MI-hívás fog valószínűleg elbukni, hogy időben átirányíthassuk egy másik modellre.

A novemberi napló 1000 hívást tartalmaz, már tisztítva (ahogy a 2. blokkban tanultuk). Két modellt építesz:
1. **osztályozás:** sikeres lesz-e a hívás (címke: `sikeres`);
2. **regresszió:** mennyi lesz a válaszidő (címke: `valaszido_ms`).

Ahol `...  # TODO` áll, oda írd a megoldást. **Mentés:** File → Save a copy in GitHub → saját repó, `main`, fájlnév `modell.ipynb`, commitüzenet: `feat: első modellek (feladatkártya #3)`.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, mean_absolute_error, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

ADAT = Path("data/hivasok_november.csv")
df = pd.read_csv(ADAT if ADAT.exists() else "https://raw.githubusercontent.com/Kizsi2025/bolyai-labs-03-modell/main/data/hivasok_november.csv")
df["modell_nagyobb"] = (df["modell"] == "nagyobb").astype(int)
print(df.shape)
print("Sikeres hívások aránya:", round(df["sikeres"].mean(), 3))
df.head()

In [ ]:
# Jellemzők (X) és címke (y) az osztályozáshoz
X = df[["tokenek", "valaszido_ms", "modell_nagyobb"]]
y = df["sikeres"]

## a) Tanító és teszthalmaz
A modell a tanítóhalmazon tanul, a teszthalmazon mérjük. A `stratify=y` miatt mindkét részben ugyanannyi a sikertelen hívások aránya.

In [ ]:
X_train, X_test, y_train, y_test = ...  # TODO: train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print(len(X_train), len(X_test))

## Alapvonal (baseline)
A legegyszerűbb „modell” mindig a leggyakoribb választ mondja: „sikeres”. Ezt kell legyőzni.

In [ ]:
alap = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
baseline = accuracy_score(y_test, alap.predict(X_test))
print("Alapvonal pontossága:", round(baseline, 3))

## b) Logisztikus regresszió
A `StandardScaler` standardizálja a jellemzőket (2. blokk: normalizálás), utána jön a modell.

In [ ]:
modell = ...  # TODO: make_pipeline(StandardScaler(), LogisticRegression())
modell.fit(X_train, y_train)
jo = modell.predict(X_test)
accuracy = accuracy_score(y_test, jo)
print("Pontosság:", round(accuracy, 3))

## c) Tévesztési mátrix és felidézés
A pontosság nem minden: azt is tudni akarjuk, a valóban elbukó hívások hány százalékát találja meg a modell. Ez a sikertelen osztály **felidézése** (recall).

In [ ]:
print(pd.crosstab(y_test, jo, rownames=["valódi"], colnames=["jósolt"]))
recall_hiba = ...  # TODO: recall_score(y_test, jo, pos_label=0)
print("A sikertelen hívások felidézése:", round(recall_hiba, 3))

## d) Regresszió: mennyi lesz a válaszidő?
Itt a címke egy szám. A hibát az átlagos abszolút eltéréssel (MAE) mérjük, ezredmásodpercben.

In [ ]:
Xr = df[["tokenek", "modell_nagyobb"]]
yr = df["valaszido_ms"]
Xr_train, Xr_test, yr_train, yr_test = train_test_split(Xr, yr, test_size=0.25, random_state=42)
reg = ...  # TODO: LinearRegression().fit(Xr_train, yr_train)
mae = mean_absolute_error(yr_test, reg.predict(Xr_test))
print("Átlagos abszolút hiba:", round(mae, 1), "ms")

## Önellenőrzés és mentés

In [ ]:
assert len(X_test) == 250, "a) A teszthalmaz 250 sor legyen (test_size=0.25)."
assert accuracy > baseline + 0.05, "b) A modell nem győzte le az alapvonalat."
assert recall_hiba > 0.5, "c) A felidézést a sikertelen (0) osztályra számold: pos_label=0."
assert mae < 60, "d) Túl nagy a regresszió hibája."
Path("eredmenyek").mkdir(exist_ok=True)
json.dump({"baseline": baseline, "accuracy": accuracy, "recall_hiba": recall_hiba, "mae": mae},
          open("eredmenyek/kartya3.json", "w"), indent=1)
print("Minden ellenőrzés rendben!", {"alapvonal": round(baseline, 3), "pontosság": round(accuracy, 3),
                                    "felidézés": round(recall_hiba, 3), "MAE": round(mae, 1)})

**Gondolkodj el (a kilépőkérdésekhez):** az alapvonal is 71%-os pontosságú. Miért nem elég a pontosságot nézni, ha a sikertelen hívásokat akarjuk elkapni?